# Getting Started

## First STLDriver object

In [ ]:
import stlrom as stl
d = stl.STLDriver()
d.parse_string('signal x,y')
print(d)

## Adding signal data

In [ ]:
d.add_sample([0, -0.4, 0.6]) # time, x, y
d.add_sample([0.5, 0.7, 0.1])
d.add_sample([1.1, -0.1, 0.7])
print(d)

## First Plots

In [ ]:
import matplotlib.pyplot as plt

ax = d.plot_signal('x')  # plots signal x, returns axis
ax = d.plot_signal('y',ax=ax) # plots signal y on the same axis

## First Predicates

In [ ]:
d.parse_string("""
mu_x    := x[t]>0
mu_y    := y[t]>.5
mu_comp := x[t]<y[t]
""")
print(d)

## Satisfaction Evaluation

### Boolean Satisfaction

In [ ]:
d.set_semantics('BOOLEAN')
sat_mu = d.eval_rob('mu_x',0,1.1)
print(sat_mu)

In [ ]:
ax = d.plot_signal('x')  # plots signal x, returns axis
ax = sat_mu.plot(label='Satisfaction of x > 0', ax=ax)


### Robust Satisfaction

In [ ]:
sat_mu_y = d.eval_rob('mu_y',0,1.1)
print(sat_mu_y)

In [ ]:
d.set_semantics('SPACE')
rob_mu_y = d.eval_rob('mu_y',0,1.1)
print(rob_mu_y)

In [ ]:
ax = d.plot_signal('y')  # plots signal x, returns axis
sat_mu_y.plot(label='Satisfaction of y > 2', ax=ax)
rob_mu_y.plot(label='Robustness of y > 2', ax=ax)


# Signal Generation

### Piecewise Constant Signal 

In [ ]:
from stlrom import PWCSignalGen

pwc_gen = PWCSignalGen(times=[0, 1., 2., 2.5, 3.],
                       values=[0., 1., -0.5, 0.8, 0.5])
pwc_signal = pwc_gen.get_signal(t0=0, tf=3)
pwc_signal.plot(label='PWC signal');

### Piecewise Linear Signals

In [ ]:
from stlrom import PWLSignalGen

pwl_gen = PWLSignalGen(times=[0, 1., 2., 2.5, 3.],
                       values=[0., 1., -0.5, 0.8, 0.5])
pwl_signal = pwl_gen.get_signal(t0=0, tf=3)
pwl_signal.plot(label='PWL signal');

### Oscillation Signals

In [ ]:
from stlrom import OscillSignalGen

osc_gen = OscillSignalGen(period=2, amplitude=0.25, base=0, damp=-0.5)
osc_signal= osc_gen.get_signal(t0=0, tf=10, dt=0.02)
osc_signal.plot(label='Oscillating signal');


# Robustness of Equality

In [ ]:
d2 = stl.STLDriver()
d2.parse_string("""
    signal x
    mu_eq := x[t] == 0
    """)

d2.set_signals([osc_signal])
print(d2)

In [ ]:
stl.Signal.set_Eps(0.01)

rob_eq = d2.eval_rob('mu_eq', 0, 10)

ax = osc_signal.plot(label='Oscillating signal');
ax.plot([0, osc_signal.end_time], [stl.Signal.get_Eps(), stl.Signal.get_Eps()], linestyle='--', color='black', label="± $\\varepsilon$")
ax.plot([0, osc_signal.end_time], [-stl.Signal.get_Eps(), -stl.Signal.get_Eps()], linestyle='--', color='black')
ax = rob_eq.plot(label='Robustness of mu_eq for oscillating signal', ax=ax);



In [ ]:
stl.Signal.set_Eps(1e-12)

# Temperature Example

## Generating Signal

In [ ]:
import numpy as np
times = np.linspace(0, 12, 100)
values = 18 + 9 * np.sin(times) + 3 * np.sin(3 * times)
temp_signal = PWLSignalGen(times = times, values = values).get_signal(tf=12)

ax = temp_signal.plot(label='Temperature signal');
ax.plot([0, temp_signal.end_time], [25, 25], linestyle='--', color='black', label="Threshold temperature");
ax.legend();


## Parsing Formulas

In [ ]:
d3 = stl.STLDriver()
d3.parse_string("""signal temp                
    param Tmax = 25, tau=2
    mu_temp := temp[t] < Tmax

    # Temperature should always be below Tmax
    phi_ev := ev[0, tau] (mu_temp)

    # If temperature is above Tmax, it should eventually be below Tmax within 2 seconds
    phi := alw[0, 8 ] ((not mu_temp) => phi_ev)
    """)

d3.set_signals([temp_signal])

## Robustness of Top Formula

In [ ]:
d3.set_param('tau', 1);d3.set_param('Tmax', 20);
d3.set_semantics('BOOLEAN')
sat_phi_ev = d3.eval_rob('phi_ev', 0,6)
d3.set_semantics('SPACE')
r_phi_ev = d3.eval_rob('phi_ev', 0, 6)

ax = temp_signal.plot(label='Temperature signal');
r_phi_ev.plot(label='Robustness of phi_ev', ax=ax);

ax_bool = ax.twinx(); ax_bool.set_yticks([0, 1]); ax_bool.set_yticklabels(['False', 'True']);
sat_phi_ev.plot(label ='Satisfaction of phi_ev', ax=ax_bool, color='green');

## Robustness Map

In [ ]:
r_phi, _, _ = d3.eval_online_rob('phi', 0, 6)
rob_map = d3.get_online_robustness_map('phi')
stl.plot_rob_map_widget(rob_map)